> ⚠️ **Before you start:** This is a read-only course copy.
> Go to **File → Save a copy in Drive** right now, then continue working in *your* copy.
> Changes made here will not be saved.

In [ ]:
# This notebook only needs boto3.
!pip install -q boto3

# Setup: AWS Credentials for Colab

## What you need

Stages 01 to 06 need nothing but a browser. Stage 07 is the first one that calls a
real model on Amazon Bedrock, so it is the first one that needs credentials.

Work through this once and stage 07 will run. Stages 08 to 10 ask for more than a
Bedrock key can give; there is a note at the end on why, and those stages will cover
their own setup when they land.

### The three gates

People usually assume "it didn't work" means one problem. It's almost always one of
three, and they fail at different moments:

| Gate | What it is | How it fails |
|------|-----------|--------------|
| **Credential** | A key that proves who you are | `UnrecognizedClientException`, or a silent `None` |
| **Region** | Bedrock is per-region | `NoRegionError` before anything hits the network |
| **Anthropic's use case form** | A one-time form, once per account | `AccessDeniedException` |

The last cell in this notebook checks all three separately and tells you which one is
failing. Run it whenever something breaks later in the course.

## Part 1: Generate a Bedrock API key

Amazon Bedrock issues its own API keys: one bearer token, generated in the Bedrock
console. Use one of these rather than your account's access key and secret. A Bedrock
API key is scoped to Bedrock and Bedrock Runtime actions, so it cannot touch S3, cannot
touch SageMaker, and cannot read your account. Worst case, someone burns model quota
you revoke in a click.

They come in two flavours:

- **Short-term:** valid up to 12 hours, inherits the permissions of whoever generated
  it. This is what AWS recommends for production.
- **Long-term:** valid until an expiry date you choose. AWS recommends these *only for
  exploration*, which is exactly what a course is.

Use a **long-term key** so you aren't regenerating it every session, and give it a short
expiry.

### Steps

1. Sign in to the [AWS console](https://console.aws.amazon.com/).
2. Go to **Amazon Bedrock** → in the left sidebar, near the bottom, **API keys**.
3. Choose the **Long-term API keys** tab → **Generate long-term API key**.
4. Set an expiry. **7 days** is plenty to get through stage 07. Pick the shortest
   window you'll actually use; you can always generate another.
5. Click generate, then **copy the key**. It's shown once. If you lose it, generate a
   new one and delete the old. There is no way to view it again.

> AWS creates an IAM user behind the scenes to carry this key, with service-specific
> credentials limited to Bedrock. You don't need to manage that user; deleting the key
> is enough.

## Part 2: Pick a region

Bedrock is regional, and which models exist varies by region. Pick one and use it for
the rest of the course.

`us-east-1` is the safe default. It gets models first and has the widest selection.

Write down whichever you pick. You need it in Part 4, and a mismatch between your region
and your model ID is the single most common reason the verify cell fails.

## Part 3: Submit Anthropic's use case form

Every Bedrock foundation model is enabled by default, so there is no list of checkboxes
to tick. Anthropic models carry one extra requirement: AWS asks for a one-time use case
form before your first call.

It is **once per account, not once per region**, and access is granted as soon as you
submit. If your account sits under an AWS Organization and someone filled the form in at
the management account, you have already inherited it and can skip this.

1. In the Bedrock console, open the **model catalog** and select any Anthropic Claude model.
2. Choose **Submit use case details**.
3. Fill in the form and submit it.

The form wants a company name, a website, and what you plan to build. If you are an
individual working through this course, AWS accepts a GitHub profile or a link to a
personal project as the URL.

Skip this and the verify cell fails with `AccessDeniedException`, which reads like a
broken key. It is the most common false alarm in this whole setup.

> **Two footnotes.** Opt-in regions need the form submitted again separately; the
> default `us-east-1` is not one. And the first call from a brand new account also
> kicks off an AWS Marketplace subscription in the background, which can take a few
> minutes to settle. If the verify cell fails and then starts working on its own
> shortly after, that was it.

## Part 4: Store the key in Colab Secrets

Colab Secrets is the environment-variable store. You set a value once in the sidebar,
grant this notebook access to it, and read it at runtime. Values live in your Google
account, so the same secret is there for every notebook in the course.

1. In the Colab sidebar, click the **🔑 key icon** (*Secrets*).
2. **+ Add new secret**. Name it exactly `AWS_BEARER_TOKEN_BEDROCK`, paste the key from
   Part 1 as the value.
3. Add a second secret named `AWS_DEFAULT_REGION`, value = the region from Part 2
   (e.g. `us-east-1`).
4. **Toggle "Notebook access" on for both.** This is per notebook and it is off by
   default. If you skip it, the next cell fails with `NotebookAccessError` even though
   the secrets exist.

Set them once and every notebook in the course can use them. You just flip the
Notebook access toggle for each new one.

### Loading them

The names matter. Those two are exactly what boto3 looks for in the environment, so once
they're loaded, no code anywhere else in the course has to know about credentials.

In [ ]:
import os

def load_aws_secrets():
    """Copy Colab Secrets into environment variables.

    boto3 reads AWS_BEARER_TOKEN_BEDROCK and AWS_DEFAULT_REGION from the
    environment on its own, so after this runs every later cell just works.
    No client is ever handed a credential explicitly.
    """
    try:
        from google.colab import userdata
    except ImportError:
        print("Not running in Colab; using whatever credentials this machine already has.")
        return

    for name in ("AWS_BEARER_TOKEN_BEDROCK", "AWS_DEFAULT_REGION"):
        try:
            os.environ[name] = userdata.get(name)
            print(f"loaded  {name}")
        except Exception as e:
            # SecretNotFoundError  -> the secret does not exist (check the spelling)
            # NotebookAccessError  -> it exists, but this notebook is not allowed to read it
            print(f"MISSING {name}  ({type(e).__name__})")


load_aws_secrets()

## Part 5: Verify

This cell checks the three gates in order and stops at the first one that fails, so the
message you get is one sentence naming the problem.

Run it now. Run it again any time a later stage stops working.

In [ ]:
import os

import boto3
import botocore.exceptions

# Bedrock reaches Claude through a cross-region inference profile, and the prefix
# has to match your region: us-east-1 -> "us.", eu-west-1 -> "eu.", and so on.
BASE_MODEL = "anthropic.claude-sonnet-4-6"

# What each Bedrock error actually means, in the order you are likely to hit them.
REMEDIES = {
    "UnrecognizedClientException": "The key is not valid. Regenerate it (Part 1) and update the secret.",
    "InvalidSignatureException": "The key is malformed, likely a partial copy/paste. Re-copy it.",
    "AccessDeniedException": (
        "Two things produce this. If the message mentions the key format, the value in "
        "your secret is not a Bedrock API key, so re-copy it (Part 1). Otherwise the key is "
        "fine and you have not submitted Anthropic's use case form yet (Part 3). A brand "
        "new account can also throw this for a few minutes while the Marketplace "
        "subscription settles."
    ),
    "ValidationException": "This model ID is not available in this region. Check the region prefix.",
    "ResourceNotFoundException": "No such model in this region. Check Part 2 and Part 3 agree.",
    "ThrottlingException": "Rate limited by Bedrock. Wait a moment and run this again.",
}


def verify():
    # Gate 1: is there a credential at all?
    if not os.environ.get("AWS_BEARER_TOKEN_BEDROCK"):
        print("FAIL  no API key in the environment.")
        print("      Run the cell in Part 4. If it printed MISSING, the secret is not")
        print("      set or Notebook access is off for it.")
        return False
    print("ok    API key is set")

    # Gate 2: Bedrock is regional, and boto3 fails before the network without one.
    region = os.environ.get("AWS_DEFAULT_REGION")
    if not region:
        print("FAIL  no region set. Add an AWS_DEFAULT_REGION secret (Part 2).")
        return False
    model_id = f"{region.split('-')[0]}.{BASE_MODEL}"
    print(f"ok    region is {region}, so the model ID is {model_id}")

    # Gate 3: the only real proof is a call that comes back.
    try:
        client = boto3.client("bedrock-runtime", region_name=region)
        response = client.converse(
            modelId=model_id,
            messages=[{"role": "user", "content": [{"text": "Reply with the single word: ready"}]}],
        )
    except botocore.exceptions.ClientError as e:
        code = e.response["Error"]["Code"]
        print(f"FAIL  Bedrock rejected the call: {code}")
        print(f"      {REMEDIES.get(code, e.response['Error'].get('Message', ''))}")
        return False
    except botocore.exceptions.BotoCoreError as e:
        print(f"FAIL  could not reach Bedrock: {type(e).__name__}: {e}")
        return False

    reply = response["output"]["message"]["content"][0]["text"].strip()
    usage = response["usage"]
    print(f"ok    Claude replied: {reply!r}")
    print(f"      ({usage['inputTokens']} in / {usage['outputTokens']} out, a fraction of a cent)")
    print()
    print("All three gates open. Stage 07 will run.")
    return True


verify()

## Part 6: Housekeeping

**Cost.** Everything in stage 07 costs a fraction of a cent, and the verify call above
is a handful of tokens. Set a
[budget alert](https://console.aws.amazon.com/billing/home#/budgets) anyway. Nothing
here will come close to tripping it; the habit is worth having before stage 08, where
the numbers stop being rounding errors.

**Rotation.** The key expires on the date you set in Part 1. When it does, generate a new
one and update the Colab secret. Nothing else changes.

**Revoking.** Bedrock console → **API keys** → select the key → delete. It stops working
immediately. Do this the moment you suspect it's out of your hands, then generate a
replacement; there is no partial disable.

## What about stages 08 to 10?

A Bedrock API key is deliberately narrow: Bedrock and nothing else. That's what makes
it safe here, and it's also why it won't carry you through the rest of the course. S3,
SageMaker, and MLflow all need credentials with real account access.

That changes the calculation. A credential that can start SageMaker training jobs is a
bad thing to keep in a browser VM you don't control, so the "run it all in Colab" model
runs out at stage 07. Stages 08 to 10 move to your own AWS account with session
credentials, and those stages will cover that setup when they land.

For now: you have what stage 07 needs.

---

**Next:** [Stage 07: LLM Integration](https://colab.research.google.com/github/marceloacosta/churn-prediction-pipeline/blob/main/modules/07-llm-integration/07-llm-integration.ipynb)

*Reference: [Bedrock API keys](https://docs.aws.amazon.com/bedrock/latest/userguide/api-keys.html), [using an API key](https://docs.aws.amazon.com/bedrock/latest/userguide/api-keys-use.html)*  
*Series: [Build with AWS](https://buildwithaws.substack.com/)*